# 22. pgvector IVFFlat — lists · probes · 분포 변화

12번에서 numpy 로 만든 것을 pgvector 로 한다.

| 시점 | 파라미터 | 뜻 | pgvector 권장 |
|---|---|---|---|
| 빌드 | `lists` | 군집 수 (= nlist) | 100만 행 이하 `rows/1000`, 초과 `√rows` |
| 검색 | `ivfflat.probes` | 뒤질 군집 수 (= nprobe) | 기본 1. `√lists` 부근부터 |

이 노트북은 IVFFlat 만 재기 위해 21번의 HNSW 인덱스를 지우고 시작한다 (23번에서 다시 만든다).

| 셀 | 하는 일 |
|---|---|
| 1 | 설정 |
| 2 | 연결 · 인덱스 정리 |
| 3 | lists × probes — recall 과 지연시간 |
| 4 | 군집이 없는 데이터라면 |
| 5 | 분포가 바뀌면 — REINDEX |

**Run → Run All Cells** 로 한 번에 실행하거나, 셀을 하나씩 **Shift + Enter**

### 셀 1 · 설정

In [ ]:
# 셀 1 · 설정
DB_URL = "postgresql://lab:lab@db:5432/lab"

### 셀 2 · 연결 · 인덱스 정리

In [ ]:
# 셀 2 · 연결 · 인덱스 정리
from labutil import *

conn = connect(DB_URL)
for ds in ["clu", "iso"]:
    conn.execute(f"DROP INDEX IF EXISTS vec_{ds}_hnsw")
    conn.execute(f"DROP INDEX IF EXISTS vec_{ds}_ivf")
N = q1(conn, "SELECT count(*) FROM vec_clu")

## 1. lists 를 바꿔 가며 빌드하고 probes 로 검색

### 셀 3 · lists × probes — recall 과 지연시간

In [ ]:
# 셀 3 · lists × probes — recall 과 지연시간
PROBES = [1, 2, 4, 8, 16, 32, 64]
rows, builds = [], []
for lists in [100, 316, 1265]:                         # rows/1000 · √N · 4√N
    conn.execute("DROP INDEX IF EXISTS vec_clu_ivf")
    with timer() as t:
        conn.execute(f"CREATE INDEX vec_clu_ivf ON vec_clu USING ivfflat (emb vector_l2_ops) WITH (lists = {lists})")
    builds.append({"lists": lists, "빌드(s)": t["s"], "인덱스 MB": rel_size(conn, "vec_clu_ivf")})
    for p in PROBES:
        conn.execute(f"SET ivfflat.probes = {p}")
        rows.append({"lists": lists, "probes": p, **pg_bench(conn, "vec_clu", "clu")})
conn.execute("RESET ivfflat.probes")
ivf = pd.DataFrame(rows)
builds = pd.DataFrame(builds)

try:
    hnsw = pd.read_csv("results/hnsw_curve.csv")       # 21번 결과
except FileNotFoundError:
    hnsw = None

fig, axes = plt.subplots(1, 2, figsize=(15, 4.8), gridspec_kw={"width_ratios": [2, 1]}, layout="constrained")
for (lists, g), color in zip(ivf.groupby("lists"), ["#9aa5a8", "#FF7043", "#3D484A"]):
    axes[0].plot(g["ms"], g["recall"], "-o", color=color, label=f"IVFFlat lists={lists}")
    for _, r in g.iterrows():
        axes[0].annotate(int(r.probes), (r.ms, r.recall), textcoords="offset points", xytext=(3, -10), fontsize=7, color=color)
if hnsw is not None:
    h = hnsw[hnsw.ds == "clu"]
    axes[0].plot(h["ms"], h["recall"], "--s", color="#009CA6", label="HNSW (21번)")
axes[0].set_xscale("log"); axes[0].set_xlabel("지연시간 ms   ※ 점 옆 숫자 = probes"); axes[0].set_ylabel("recall@10")
axes[0].legend(); axes[0].set_title("군집형 10만 개: probes ↑ → recall ↑ · 느려짐")
axes[1].bar(builds["lists"].astype(str), builds["빌드(s)"], color="#3D484A")
for i, r in builds.iterrows():
    axes[1].text(i, r["빌드(s)"], f"{r['빌드(s)']:.1f}s\n{r['인덱스 MB']:.0f}MB", ha="center", va="bottom", fontsize=9)
axes[1].set_xlabel("lists"); axes[1].set_title("빌드 시간 · 크기"); axes[1].set_ylim(0, builds["빌드(s)"].max() * 1.3)
plt.show()
ivf.pivot(index="probes", columns="lists", values="recall").round(3)

**관찰:**
- `probes=1`(기본값)은 빠르지만 recall 이 낮다. 기본값 그대로 쓰면 안 되는 이유
- `lists` 가 클수록 같은 probes 에서 recall 은 낮고 빠르다 — 군집 하나가 작기 때문
- **이 데이터에서는 IVFFlat 이 HNSW 보다 낫다.** lists=316·probes=4 가 recall 1.0 인데 HNSW 기본값은 약 0.9.
  군집 200개가 뚜렷한 데이터라 k-means 중심이 진짜 군집과 거의 일치하기 때문 — 슬라이드 "범주가 뚜렷하면 IVF"
- IVFFlat 빌드는 lists 가 작으면 1초 안쪽으로 HNSW(21번 약 8초)보다 빠르고, 인덱스도 작다 (벡터 + 군집 번호뿐).
  lists=1265 는 k-means 비용(N·k·d·T)이 커져 빌드가 가장 오래 걸린다
- 지연시간은 1ms 안팎이라 네트워크 왕복과 잡음이 섞여 있다. 순서만 보고 절대값은 믿지 말 것

## 2. 군집이 없는 데이터라면? — "범주가 뚜렷하면 IVF" 검증
같은 설정(lists=316)을 무구조형 데이터에 적용한다.

### 셀 4 · 군집이 없는 데이터라면

In [ ]:
# 셀 4 · 군집이 없는 데이터라면
conn.execute("DROP INDEX IF EXISTS vec_clu_ivf")
conn.execute("CREATE INDEX vec_clu_ivf ON vec_clu USING ivfflat (emb vector_l2_ops) WITH (lists = 316)")
conn.execute("CREATE INDEX vec_iso_ivf ON vec_iso USING ivfflat (emb vector_l2_ops) WITH (lists = 316)")
rows = []
for ds in ["clu", "iso"]:
    for p in PROBES:
        conn.execute(f"SET ivfflat.probes = {p}")
        rows.append({"ds": ds, "probes": p, **pg_bench(conn, f"vec_{ds}", ds)})
conn.execute("RESET ivfflat.probes")
cmp_ = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.4), layout="constrained")
for ds, color, name in [("clu", "#FF7043", "군집형"), ("iso", "#3D484A", "무구조형")]:
    g = cmp_[cmp_.ds == ds]
    axes[0].plot(g["probes"], g["recall"], "-o", color=color, label=f"IVFFlat · {name}")
    if hnsw is not None:
        h = hnsw[hnsw.ds == ds]
        axes[1].plot(h["ms"], h["recall"], "--s", color=color, alpha=0.6, label=f"HNSW · {name}")
    axes[1].plot(g["ms"], g["recall"], "-o", color=color, label=f"IVFFlat · {name}")
axes[0].set_xscale("log", base=2); axes[0].set_xlabel("probes (lists=316)"); axes[0].set_ylabel("recall@10")
axes[0].legend(); axes[0].set_title("같은 probes 에서 recall")
axes[1].set_xscale("log"); axes[1].set_xlabel("지연시간 ms"); axes[1].set_ylabel("recall@10")
axes[1].legend(fontsize=8); axes[1].set_title("HNSW vs IVFFlat, 데이터별")
plt.show()
cmp_.pivot(index="probes", columns="ds", values="recall").round(3)

**관찰:**
- 무구조형에서는 IVFFlat 이 무너진다(probes=64 로 316개 중 64개 군집을 뒤져도 약 0.7). 군집이 없으니 "가까운 군집"이 곧 "가까운 이웃"이 아니다
- HNSW 도 무구조형에서 떨어지지만, 같은 지연시간에서 IVFFlat 보다 recall 이 높다
- 결론: **데이터 구조에 따라 승자가 바뀐다.** 실제 임베딩은 두 극단의 중간 어딘가 → 슬라이드의 선택 가이드는 출발점이고, 결정은 내 데이터로 잰 곡선으로 한다

## 3. 데이터 분포가 바뀌면 — 중심이 낡는다
군집 200개 중 **앞의 100개만 있을 때** 인덱스를 만들고, 나중에 나머지 100개 군집의 데이터를 넣는다.
IVFFlat 은 INSERT 때 새 벡터를 **기존 중심 중 가장 가까운 리스트**에 넣을 뿐 중심을 다시 학습하지 않는다.

### 셀 5 · 분포가 바뀌면 — REINDEX

In [ ]:
# 셀 5 · 분포가 바뀌면 — REINDEX
conn.execute("DROP TABLE IF EXISTS vec_shift")
conn.execute("CREATE TABLE vec_shift AS SELECT * FROM vec_clu WHERE cluster < 100")
conn.execute("CREATE INDEX vec_shift_ivf ON vec_shift USING ivfflat (emb vector_l2_ops) WITH (lists = 316)")
conn.execute("INSERT INTO vec_shift SELECT * FROM vec_clu WHERE cluster >= 100")   # 새 분포 유입
conn.execute("ANALYZE vec_shift")

# 질의를 "정답이 옛 군집에 있는 것" / "새 군집에 있는 것" 으로 나눈다
grp = q(conn, """SELECT g.qid, CASE WHEN c.cluster < 100 THEN '옛 군집' ELSE '새 군집' END AS grp
                 FROM vec_gt g JOIN vec_clu c ON c.id = g.id WHERE g.ds = 'clu' AND g.rank = 1""").set_index("qid")["grp"]


def split_recall(table, probes=8):
    conn.execute(f"SET ivfflat.probes = {probes}")
    qs = q(conn, "SELECT qid, emb FROM vec_query WHERE ds='clu' ORDER BY qid")
    gt = q(conn, "SELECT qid, array_agg(id ORDER BY rank) ids FROM vec_gt WHERE ds='clu' AND rank<=10 GROUP BY qid").set_index("qid")["ids"]
    out = {"옛 군집": [], "새 군집": []}
    for qid, emb in qs.itertuples(index=False):
        ids = q(conn, f"SELECT id FROM {table} ORDER BY emb <-> %s LIMIT 10", (emb,))["id"].tolist()
        out[grp[qid]].append(len(set(ids) & set(gt[qid])) / 10)
    conn.execute("RESET ivfflat.probes")
    return {k: float(np.mean(v)) for k, v in out.items()}


before = split_recall("vec_shift")
with timer("REINDEX (중심 재학습)"):
    conn.execute("REINDEX INDEX vec_shift_ivf")
after = split_recall("vec_shift")

fig, ax = plt.subplots(figsize=(8, 3.8))
x = np.arange(2)
ax.bar(x - 0.2, [before[k] for k in ["옛 군집", "새 군집"]], 0.4, color="#9aa5a8", label="절반만 보고 만든 인덱스")
ax.bar(x + 0.2, [after[k] for k in ["옛 군집", "새 군집"]], 0.4, color="#FF7043", label="REINDEX 후")
for i, k in enumerate(["옛 군집", "새 군집"]):
    ax.text(i - 0.2, before[k], f"{before[k]:.2f}", ha="center", va="bottom")
    ax.text(i + 0.2, after[k], f"{after[k]:.2f}", ha="center", va="bottom")
ax.set_xticks(x, ["정답이 옛 군집에 있는 질의", "정답이 새 군집에 있는 질의"]); ax.set_ylim(0, 1.3)
ax.set_ylabel("recall@10 (probes=8)"); ax.legend(loc="upper center", ncol=2); ax.set_title("분포가 바뀌면 중심 재학습이 필요하다")
plt.show()
conn.execute("DROP TABLE vec_shift")
for ds in ["clu", "iso"]:                         # 뒷 노트북을 위해 IVFFlat 인덱스 정리
    _ = conn.execute(f"DROP INDEX IF EXISTS vec_{ds}_ivf")

**관찰:** 새로 들어온 군집의 데이터는 엉뚱한 옛 중심들에 흩어져 들어가서, 그 근처 질의의 recall 이 떨어진다.
`REINDEX` 로 전체 데이터에서 중심을 다시 학습하면 회복된다 (슬라이드 "분포가 바뀌면 centroid 재학습").
같은 이유로 **빈 테이블에 IVFFlat 을 먼저 만들고 데이터를 넣으면 안 된다** — 데이터를 넣은 뒤 만든다.